# NERMuD 2023 — Data Preprocessing

This notebook prepares the NERMuD 2023 dataset for Named Entity Recognition with Transformer models.

The preprocessing pipeline consists of:

1. Load raw NERMuD
2. Inspect dataset
3. Data quality checks
   └── entity/text mismatches
4. Resolve annotation matching issues
5. Tokenization
6. BIO encoding
7. Validation
8. Save processed datasets

In [2]:
import json
import re
from pathlib import Path
from collections import Counter

import pandas as pd

In [3]:
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
train_files = {
    "adg": RAW_DIR / "adg_train.jsonl",
    "fic": RAW_DIR / "fic_train.jsonl",
    "wn": RAW_DIR / "wn_train.jsonl"
}

dev_files = {
    "adg": RAW_DIR / "adg_dev.jsonl",
    "fic": RAW_DIR / "fic_dev.jsonl",
    "wn": RAW_DIR / "wn_dev.jsonl"
}

test_files = {
    "adg": RAW_DIR / "adg_test.jsonl",
    "fic": RAW_DIR / "fic_test.jsonl",
    "wn": RAW_DIR / "wn_test.jsonl"
}

def load_jsonl(filepath, domain):
    data = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            sample = json.loads(line)
            sample["domain"] = domain
            data.append(sample)

    return data

In [5]:
train_data = []

for domain, filepath in train_files.items():
    domain_data = load_jsonl(filepath, domain)
    train_data.extend(domain_data)

print(f"Total training samples: {len(train_data):,}")

Total training samples: 27,482


In [6]:
dev_data = []

for domain, filepath in dev_files.items():
    domain_data = load_jsonl(filepath, domain)
    dev_data.extend(domain_data)

print(f"Total development samples: {len(dev_data):,}")

Total development samples: 4,767


In [7]:
test_data = []

for domain, filepath in test_files.items():
    domain_data = load_jsonl(filepath, domain)
    test_data.extend(domain_data)

print(f"Total testing samples: {len(test_data):,}")

Total testing samples: 4,126


In [25]:
train_domain_counts = Counter(
    sample["domain"]
    for sample in train_data
)

dev_domain_counts = Counter(
    sample["domain"]
    for sample in dev_data
)

test_domain_counts = Counter(
    sample["domain"]
    for sample in test_data
)

print("Training set:")
for domain, count in train_domain_counts.items():
    print(f"  {domain}: {count:,}")

print("\nDevelopment set:")
for domain, count in dev_domain_counts.items():
    print(f"  {domain}: {count:,}")
    
print("\nTesting set:")
for domain, count in test_domain_counts.items():
    print(f"  {domain}: {count:,}")

Training set:
  adg: 5,147
  fic: 11,423
  wn: 10,912

Development set:
  adg: 1,122
  fic: 1,051
  wn: 2,594

Testing set:
  adg: 521
  fic: 1,517
  wn: 2,088


In [9]:
for domain in ["adg", "fic", "wn"]:
    sample = next(
        sample for sample in train_data
        if sample["domain"] == domain
    )

    print(f"\n--- {domain.upper()} ---")
    print("Text:", sample["text"])
    print("Entities:", sample["entities"])


--- ADG ---
Text: Il ministro degli Esteri al commissario capo della Commissione alleata
Entities: [{'entity_text': 'Commissione alleata', 'type': 'ORG'}]

--- FIC ---
Text: Ustica, 19 XII 1926
Entities: [{'entity_text': 'Ustica', 'type': 'LOC'}]

--- WN ---
Text: In fiamme l' Istituto Lama Tzong Khapa, monastero buddhista
Entities: [{'entity_text': 'Istituto Lama Tzong Khapa', 'type': 'LOC'}]


In [10]:
def find_exact_match(text, entity_text, start=0):
    return text.find(entity_text, start)

In [11]:
missing_entities = []

for sample_index, sample in enumerate(train_data):

    text = sample["text"]

    for entity in sample["entities"]:

        entity_text = entity["entity_text"]

        start = find_exact_match(text, entity_text)

        if start == -1:
            missing_entities.append({
                "sample_index": sample_index,
                "domain": sample["domain"],
                "text": text,
                "entity_text": entity_text,
                "type": entity["type"]
            })

print(f"Total entity annotations: {sum(len(s['entities']) for s in train_data):,}")
print(f"Exact matches missing: {len(missing_entities):,}")

Total entity annotations: 21,815
Exact matches missing: 31


In [12]:
missing_df = pd.DataFrame(missing_entities)

missing_df[
    ["sample_index", "domain", "entity_text", "type", "text"]
]

,sample_index,domain,entity_text,type,text
0,9300,fic,Avanti !,ORG,Merlino nega( vedi l' Avanti! del 9 marzo) che...
1,9317,fic,Avanti !,ORG,Ma senta un po' i suoi amici dell' Avanti!
2,9327,fic,Avanti !,ORG,nell' articolo sull' Avanti! già si dice che i...
3,10066,fic,"112 , High Street",LOC,"112, High Street, Islington N. London."
4,17797,wn,Sanford C. Bernstein & Co.,ORG,L' analista finanziario della banca Sanford C....
5,18655,wn,Denver - Colorado Springs,LOC,È stata chiusa l' autostrada Denver- Colorado ...
6,19576,wn,"Ospedale "" Molinette """,LOC,Nonostante il tempestivo intervento del person...
7,19850,wn,Square Enix & Luminous Productions,ORG,Rift Apart Project Athia( Square Enix& Luminou...
8,19852,wn,Bethesda Softworks & Tango Gameworks,ORG,Soulstorm( Oddworld Inhabitants) GhostWire- To...
9,19852,wn,Superbrothers & Pine Scented,ORG,Soulstorm( Oddworld Inhabitants) GhostWire- To...


In [13]:
for _, row in missing_df.iterrows():

    print("=" * 80)
    print(f"Domain: {row['domain']}")
    print(f"Entity: {row['entity_text']!r}")
    print(f"Type: {row['type']}")
    print(f"Text: {row['text']}")

Domain: fic
Entity: 'Avanti !'
Type: ORG
Text: Merlino nega( vedi l' Avanti! del 9 marzo) che la lotta politica parlamentare sia contraria ai principi socialisti-anarchici.
Domain: fic
Entity: 'Avanti !'
Type: ORG
Text: Ma senta un po' i suoi amici dell' Avanti!
Domain: fic
Entity: 'Avanti !'
Type: ORG
Text: nell' articolo sull' Avanti! già si dice che i deputati possono fare tante belle cose che sarebbe veramente un tradimento il rifiutarci a fare anche noi la nostra parte.
Domain: fic
Entity: '112 , High Street'
Type: LOC
Text: 112, High Street, Islington N. London.
Domain: wn
Entity: 'Sanford C. Bernstein & Co.'
Type: ORG
Text: L' analista finanziario della banca Sanford C. Bernstein& Co. Tim Anderson, a seguito di una intervista con Ian Read, nuovo AD di Pfizer, prospetta la possibilità che la multinazionale farmaceutica, nota al grande pubblico come produttrice del Viagra, possa cedere ad altri le attività non ritenute più strategiche.
Domain: wn
Entity: 'Denver - Colorado Springs

In [14]:
def normalize_for_matching(text):
    """
    Normalize whitespace and spacing around punctuation for entity matching.

    This function is only used to locate entity spans.
    The original text is never modified.
    """

    # Normalize consecutive whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove whitespace before common punctuation
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)

    # Normalize whitespace around hyphens
    text = re.sub(r"\s*-\s*", "-", text)

    return text.strip()

def find_normalized_match(text, entity_text, search_start=0):
    """
    Find an entity in the original text while allowing differences
    in whitespace around punctuation and hyphens.

    Returns the corresponding (start, end) span in the original text.
    """

    entity = normalize_for_matching(entity_text)

    # Search through possible starting positions in the original text.
    for start in range(search_start, len(text)):

        original_index = start
        entity_index = 0

        while (
            original_index < len(text)
            and entity_index < len(entity)
        ):

            original_char = text[original_index]
            entity_char = entity[entity_index]

            # Ignore whitespace in the original text when appropriate
            if original_char.isspace():
                original_index += 1
                continue

            # Ignore whitespace in the normalized entity
            if entity_char.isspace():
                entity_index += 1
                continue

            if original_char != entity_char:
                break

            original_index += 1
            entity_index += 1

        if entity_index == len(entity):
            return start, original_index

    return None

In [15]:
resolved = []
unresolved = []

for item in missing_entities:

    result = find_normalized_match(
        item["text"],
        item["entity_text"]
    )

    if result is None:
        unresolved.append(item)
    else:
        resolved.append({
            **item,
            "start": result[0],
            "end": result[1]
        })

print(f"Previously missing: {len(missing_entities)}")
print(f"Resolved: {len(resolved)}")
print(f"Still unresolved: {len(unresolved)}")

Previously missing: 31
Resolved: 31
Still unresolved: 0


In [16]:
for item in resolved:
    start = item["start"]
    end = item["end"]

    print("=" * 80)
    print("Entity:", repr(item["entity_text"]))
    print("Original span:", repr(item["text"][start:end]))
    print("Type:", item["type"])

Entity: 'Avanti !'
Original span: ' Avanti!'
Type: ORG
Entity: 'Avanti !'
Original span: ' Avanti!'
Type: ORG
Entity: 'Avanti !'
Original span: ' Avanti!'
Type: ORG
Entity: '112 , High Street'
Original span: '112, High Street'
Type: LOC
Entity: 'Sanford C. Bernstein & Co.'
Original span: ' Sanford C. Bernstein& Co.'
Type: ORG
Entity: 'Denver - Colorado Springs'
Original span: ' Denver- Colorado Springs'
Type: LOC
Entity: 'Ospedale " Molinette "'
Original span: ' Ospedale" Molinette"'
Type: LOC
Entity: 'Square Enix & Luminous Productions'
Original span: ' Square Enix& Luminous Productions'
Type: ORG
Entity: 'Bethesda Softworks & Tango Gameworks'
Original span: ' Bethesda Softworks& Tango Gameworks'
Type: ORG
Entity: 'Superbrothers & Pine Scented'
Original span: ' Superbrothers& Pine Scented'
Type: ORG
Entity: 'Counterplay & Gearbox Software'
Original span: ' Counterplay& Gearbox Software'
Type: ORG
Entity: 'Visual Concepts & 2K Games'
Original span: ' Visual Concepts& 2K Games'
Type: OR

In [17]:
def tokenize_with_offsets(text):
    tokens = []

    for match in re.finditer(r"\w+|[^\w\s]", text, re.UNICODE):
        tokens.append({
            "text": match.group(),
            "start": match.start(),
            "end": match.end()
        })

    return tokens

In [18]:
text = "Sono arrivato a Roma."

tokens = tokenize_with_offsets(text)

tokens

[{'text': 'Sono', 'start': 0, 'end': 4},
 {'text': 'arrivato', 'start': 5, 'end': 13},
 {'text': 'a', 'start': 14, 'end': 15},
 {'text': 'Roma', 'start': 16, 'end': 20},
 {'text': '.', 'start': 20, 'end': 21}]

In [19]:
def create_bio_labels(sample):
    text = sample["text"]
    entities = sample["entities"]

    tokens = tokenize_with_offsets(text)
    labels = ["O"] * len(tokens)

    search_start = 0

    for entity in entities:

        entity_text = entity["entity_text"]
        entity_type = entity["type"]

        # First try exact matching
        start = text.find(entity_text, search_start)

        # If exact matching fails, use normalized matching
        if start == -1:

            result = find_normalized_match(
                text,
                entity_text,
                search_start
            )

            if result is None:
                raise ValueError(
                    f"Could not locate entity: {entity_text!r}"
                )

            start, end = result

        else:
            end = start + len(entity_text)

        # Find tokens overlapping with the entity span
        entity_token_indices = []

        for i, token in enumerate(tokens):

            token_start = token["start"]
            token_end = token["end"]

            if token_start < end and token_end > start:
                entity_token_indices.append(i)

        # Assign BIO labels
        for position, token_index in enumerate(entity_token_indices):

            if position == 0:
                labels[token_index] = f"B-{entity_type}"
            else:
                labels[token_index] = f"I-{entity_type}"

        search_start = end

    return {
        "tokens": [token["text"] for token in tokens],
        "labels": labels,
        "domain": sample["domain"]
    }

In [20]:
bio_data = [
    create_bio_labels(sample)
    for sample in train_data
]

print(f"Processed samples: {len(bio_data):,}")

Processed samples: 27,482


In [21]:
original_entities = sum(
    len(sample["entities"])
    for sample in train_data
)

bio_entities = sum(
    sum(label.startswith("B-") for label in sample["labels"])
    for sample in bio_data
)

print(f"Original entities: {original_entities:,}")
print(f"BIO entities: {bio_entities:,}")

Original entities: 21,815
BIO entities: 21,815


In [22]:
def process_dataset(data):
    return [
        create_bio_labels(sample)
        for sample in data
    ]

In [26]:
train_bio = process_dataset(train_data)
dev_bio = process_dataset(dev_data)
test_bio = process_dataset(test_data)

In [27]:
def save_jsonl(data, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        for sample in data:
            f.write(json.dumps(sample, ensure_ascii=False) + "\n")
            
save_jsonl(
    train_bio,
    PROCESSED_DIR / "train.jsonl"
)

save_jsonl(
    dev_bio,
    PROCESSED_DIR / "dev.jsonl"
)

save_jsonl(
    test_bio,
    PROCESSED_DIR / "test.jsonl"
)
